# MJ AI Assistant — Entity Extractor
### BIO-scheme NER for extracting query, app, email, task, repo, file from commands
**Output:** `exports/mj_entity_model/`

In [ ]:
# Cell 1 — Install Required Packages
import subprocess, sys

packages = [
    "transformers>=4.40.0",
    "datasets>=2.18.0",
    "torch>=2.0.0",
    "accelerate>=0.27.0",
    "scikit-learn>=1.3.0",
    "seqeval>=1.2.2",
    "numpy",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + packages)
print("All packages installed OK")


In [ ]:
# Cell 2 — Generate Entity Extraction Dataset
import random, re, json
from collections import Counter
random.seed(42)

ENTITY_TYPES = ["query", "app_name", "email", "task", "repo", "file", "url"]
LABELS    = ["O"] + ["B-" + e for e in ENTITY_TYPES] + ["I-" + e for e in ENTITY_TYPES]
LABEL2ID  = {l: i for i, l in enumerate(LABELS)}
ID2LABEL  = {i: l for l, i in LABEL2ID.items()}
print("BIO Labels:", LABELS)

QUERIES = ["trending kannada songs","yash toxic trailer","python tutorial",
           "VTU results","lofi music beats","cricket highlights",
           "machine learning course","bollywood hits 2024","react hooks tutorial"]
APPS    = ["VS Code","Notepad","Chrome","Spotify","Discord","Slack","Zoom","VLC","Calculator"]
EMAILS  = ["john@gmail.com","boss@company.com","professor@vtu.edu","hr@company.org","client@startup.io"]
TASKS   = ["fix the login bug","write unit tests","update README",
           "pay electricity bill","buy groceries","submit assignment","prepare presentation"]
REPOS   = ["mj-assistant","my-portfolio","ai-chatbot","fastapi-backend","react-dashboard"]
FILES   = ["resume.pdf","report.pdf","project.pdf","invoice.pdf","notes.pdf"]
URLS    = ["github.com/user/repo","vtu.ac.in/results","google.com","youtube.com/trending"]

ETEMPLATES = [
    ("open youtube and search {query}", "query",    QUERIES),
    ("search {query} on youtube",       "query",    QUERIES),
    ("youtube {query}",                 "query",    QUERIES),
    ("play {query} on youtube",         "query",    QUERIES),
    ("find {query} on youtube",         "query",    QUERIES),
    ("google {query}",                  "query",    QUERIES),
    ("search {query} on google",        "query",    QUERIES),
    ("look up {query}",                 "query",    QUERIES),
    ("find information about {query}",  "query",    QUERIES),
    ("open {app_name}",                 "app_name", APPS),
    ("launch {app_name}",               "app_name", APPS),
    ("start {app_name}",                "app_name", APPS),
    ("open the {app_name} app",         "app_name", APPS),
    ("can you open {app_name}",         "app_name", APPS),
    ("send an email to {email}",        "email",    EMAILS),
    ("email {email}",                   "email",    EMAILS),
    ("compose email to {email}",        "email",    EMAILS),
    ("write an email to {email}",       "email",    EMAILS),
    ("create a task to {task}",         "task",     TASKS),
    ("add task {task}",                 "task",     TASKS),
    ("remind me to {task}",             "task",     TASKS),
    ("delete task {task}",              "task",     TASKS),
    ("mark task {task} as done",        "task",     TASKS),
    ("push to {repo}",                  "repo",     REPOS),
    ("git push {repo}",                 "repo",     REPOS),
    ("create a github repo called {repo}", "repo",  REPOS),
    ("pull from {repo}",                "repo",     REPOS),
    ("analyze {file}",                  "file",     FILES),
    ("upload {file}",                   "file",     FILES),
    ("read pdf {file}",                 "file",     FILES),
    ("open {url}",                      "url",      URLS),
    ("navigate to {url}",               "url",      URLS),
]

def bio_record(template, ent_key, ent_val):
    ph = "{" + ent_key + "}"
    if ph not in template:
        return None
    text   = template.replace(ph, ent_val)
    words  = text.split()
    e_words = ent_val.lower().split()
    wlow   = [w.lower().strip(".,!?") for w in words]
    bio    = ["O"] * len(words)
    for i in range(len(wlow) - len(e_words) + 1):
        if wlow[i:i+len(e_words)] == e_words:
            bio[i] = "B-" + ent_key
            for j in range(1, len(e_words)):
                bio[i+j] = "I-" + ent_key
            break
    return {"text": text, "tokens": words,
            "ner_tags": [LABEL2ID[l] for l in bio],
            "bio_labels": bio, ent_key: ent_val}

PER_TEMPLATE = 70
records_ent = []
for tmpl, ek, pool in ETEMPLATES:
    c = 0
    while c < PER_TEMPLATE:
        val = random.choice(pool)
        rec = bio_record(tmpl, ek, val)
        if rec:
            records_ent.append(rec)
            c += 1

random.shuffle(records_ent)
print(f"Generated {len(records_ent)} entity records")
dist = Counter(tag for r in records_ent for tag in r["bio_labels"] if tag != "O")
for lbl, cnt in sorted(dist.items()):
    print(f"  {lbl:<20} {cnt}")


In [ ]:
# Cell 3 — Save Entity Dataset as JSONL
import json
from pathlib import Path

ds_dir = Path("datasets"); ds_dir.mkdir(exist_ok=True)
out    = ds_dir / "mj_entities.jsonl"
with open(out, "w", encoding="utf-8") as f:
    for r in records_ent:
        f.write(json.dumps({"tokens": r["tokens"], "ner_tags": r["ner_tags"],
                             "text": r["text"]}, ensure_ascii=False) + "\n")
print(f"Saved {len(records_ent)} entity records -> {out}")


In [ ]:
# Cell 4 — Build HuggingFace Dataset + Split
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split

all_tok = [r["tokens"]   for r in records_ent]
all_tag = [r["ner_tags"] for r in records_ent]

tr_tok, vl_tok, tr_tag, vl_tag = train_test_split(
    all_tok, all_tag, test_size=0.15, random_state=42
)
def mk(toks, tags): return Dataset.from_dict({"tokens": toks, "ner_tags": tags})
hf_ent = DatasetDict({"train": mk(tr_tok, tr_tag), "validation": mk(vl_tok, vl_tag)})
print(f"Train: {len(tr_tok)}  | Val: {len(vl_tok)}")


In [ ]:
# Cell 5 — Tokenize & Align BIO Labels
from transformers import AutoTokenizer

MODEL_NAME = "distilbert-base-uncased"
MAX_LEN    = 64
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)

def tok_align(examples):
    tok_out = tokenizer(examples["tokens"], truncation=True,
                        is_split_into_words=True, max_length=MAX_LEN)
    all_labels = []
    for i, tags in enumerate(examples["ner_tags"]):
        wids = tok_out.word_ids(batch_index=i)
        aligned, prev = [], None
        for wid in wids:
            if wid is None:
                aligned.append(-100)
            elif wid != prev:
                aligned.append(tags[wid])
            else:
                raw = LABELS[tags[wid]]
                aligned.append(LABEL2ID["I-" + raw[2:]] if raw.startswith("B-") else tags[wid])
            prev = wid
        all_labels.append(aligned)
    tok_out["labels"] = all_labels
    return tok_out

hf_ent_tok = hf_ent.map(tok_align, batched=True, remove_columns=["tokens", "ner_tags"])
print("Tokenization complete:", list(hf_ent_tok["train"].features.keys()))


In [ ]:
# Cell 6 — Train DistilBERT NER Model
import torch, numpy as np
from transformers import (
    AutoModelForTokenClassification, TrainingArguments,
    Trainer, DataCollatorForTokenClassification,
)
from seqeval.metrics import classification_report as seq_rep

BATCH  = 32
EPOCHS = 5
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Training on: {device.upper()}")

ner_model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME, num_labels=len(LABELS), id2label=ID2LABEL, label2id=LABEL2ID,
)
collator = DataCollatorForTokenClassification(tokenizer)

def ner_metrics(ep):
    pids, lbls = np.argmax(ep.predictions, axis=-1), ep.label_ids
    ts, ps = [], []
    for pr, lb in zip(pids, lbls):
        t, p = [], []
        for a, b in zip(pr, lb):
            if b == -100: continue
            t.append(ID2LABEL[b]); p.append(ID2LABEL[a])
        ts.append(t); ps.append(p)
    r = seq_rep(ts, ps, output_dict=True, zero_division=0)
    return {"f1": r.get("weighted avg", {}).get("f1-score", 0)}

from pathlib import Path
chk = Path("models/ner_checkpoints"); chk.mkdir(parents=True, exist_ok=True)

ner_args = TrainingArguments(
    output_dir=str(chk), num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH, per_device_eval_batch_size=64,
    learning_rate=2e-5, weight_decay=0.01,
    evaluation_strategy="epoch", save_strategy="epoch",
    load_best_model_at_end=True, metric_for_best_model="f1",
    logging_steps=50, seed=42, fp16=torch.cuda.is_available(), report_to="none",
)
ner_trainer = Trainer(
    model=ner_model, args=ner_args,
    train_dataset=hf_ent_tok["train"], eval_dataset=hf_ent_tok["validation"],
    tokenizer=tokenizer, data_collator=collator, compute_metrics=ner_metrics,
)
print("Starting NER training ...")
ner_trainer.train()
print("Done!")


In [ ]:
# Cell 7 — Evaluate Entity Extractor
from seqeval.metrics import classification_report as seq_rep
import numpy as np

pout = ner_trainer.predict(hf_ent_tok["validation"])
pids = np.argmax(pout.predictions, axis=-1)
lbls = pout.label_ids

ts, ps = [], []
for pr, lb in zip(pids, lbls):
    t, p = [], []
    for a, b in zip(pr, lb):
        if b == -100: continue
        t.append(ID2LABEL[b]); p.append(ID2LABEL[a])
    ts.append(t); ps.append(p)

print(seq_rep(ts, ps, zero_division=0))


In [ ]:
# Cell 8 — Save Entity Model
import json
from pathlib import Path

save_dir = Path("exports/mj_entity_model")
save_dir.mkdir(parents=True, exist_ok=True)

ner_trainer.save_model(str(save_dir))
tokenizer.save_pretrained(str(save_dir))

with open(save_dir / "label_mapping.json", "w") as f:
    json.dump({"label2id": LABEL2ID,
               "id2label": {str(k): v for k, v in ID2LABEL.items()},
               "entity_types": ENTITY_TYPES, "labels": LABELS}, f, indent=2)

print(f"Entity model saved -> {save_dir}")


In [ ]:
# Cell 9 — Entity Extractor Function
import torch

def extract_entities(text):
    words  = text.split()
    inputs = tokenizer(words, return_tensors="pt", truncation=True,
                       is_split_into_words=True, max_length=MAX_LEN)
    with torch.no_grad():
        pids = torch.argmax(ner_model(**inputs).logits, dim=-1)[0].tolist()

    wids = inputs.word_ids()
    ents, cur_e, cur_t, prev_w = {}, None, [], None
    for pid, wid in zip(pids, wids):
        if wid is None: continue
        lbl = ID2LABEL[pid]
        if lbl.startswith("B-"):
            if cur_e and cur_t: ents[cur_e] = " ".join(cur_t)
            cur_e = lbl[2:]; cur_t = [words[wid]] if wid != prev_w else []
        elif lbl.startswith("I-") and cur_e == lbl[2:]:
            if wid != prev_w: cur_t.append(words[wid])
        else:
            if cur_e and cur_t: ents[cur_e] = " ".join(cur_t)
            cur_e = None; cur_t = []
        prev_w = wid
    if cur_e and cur_t: ents[cur_e] = " ".join(cur_t)
    return {"text": text, "entities": ents}

tests = [
    "open youtube and search yash toxic trailer",
    "google VTU results",
    "send an email to boss@company.com",
    "create a task fix the login bug",
    "push to mj-assistant",
    "open VS Code",
    "analyze resume.pdf",
]
print("Entity Extraction Results:")
print("-" * 60)
for t in tests:
    r = extract_entities(t)
    print(f"  {t}")
    print(f"  -> {r['entities']}")
    print()


In [ ]:
# Cell 10 — Combined MJ Brain Output (Intent + Entities)
import json

def mj_brain(text):
    ents = extract_entities(text)["entities"]
    return {"text": text, "entities": ents}

commands = [
    "open youtube and search trending kannada songs",
    "google VTU results",
    "send email to professor@vtu.edu",
    "create a task fix the login bug",
    "push to mj-assistant on github",
    "open VS Code",
]
print("MJ Brain Output:")
print("=" * 60)
for cmd in commands:
    r = mj_brain(cmd)
    print(f"Input:   {cmd}")
    print(f"Entities: {json.dumps(r['entities'])}")
    print("-" * 60)


In [ ]:
# Cell 11 — Export to MJ Backend
import shutil
from pathlib import Path

src = Path("exports/mj_entity_model")
dst = Path("../backend/app/ml_models/mj_entity_model")
dst.mkdir(parents=True, exist_ok=True)

for f in src.iterdir():
    shutil.copy2(f, dst / f.name)

print(f"Entity model exported: {dst.resolve()}")
for f in sorted(dst.iterdir()):
    print(f"  {f.name}")


In [ ]:
# Cell 12 — FastAPI Integration Code

CODE = """
# app/ml/entity_extractor.py

import json, torch
from functools import lru_cache
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForTokenClassification

MODEL_PATH = Path(__file__).parent.parent / "ml_models" / "mj_entity_model"

@lru_cache(maxsize=1)
def _load():
    tok   = AutoTokenizer.from_pretrained(str(MODEL_PATH))
    model = AutoModelForTokenClassification.from_pretrained(str(MODEL_PATH))
    model.eval()
    with open(MODEL_PATH / "label_mapping.json") as f:
        m = json.load(f)
    return tok, model, {int(k): v for k, v in m["id2label"].items()}

def extract_entities(text: str) -> dict:
    tok, model, id2label = _load()
    words = text.split()
    inp   = tok(words, return_tensors="pt", truncation=True,
                is_split_into_words=True, max_length=64)
    with torch.no_grad():
        pids = torch.argmax(model(**inp).logits, dim=-1)[0].tolist()
    wids = inp.word_ids()
    ents, cur_e, cur_t, prev = {}, None, [], None
    for pid, wid in zip(pids, wids):
        if wid is None: continue
        lbl = id2label[pid]
        if lbl.startswith("B-"):
            if cur_e and cur_t: ents[cur_e] = " ".join(cur_t)
            cur_e = lbl[2:]; cur_t = [words[wid]] if wid != prev else []
        elif lbl.startswith("I-") and cur_e == lbl[2:]:
            if wid != prev: cur_t.append(words[wid])
        else:
            if cur_e and cur_t: ents[cur_e] = " ".join(cur_t)
            cur_e = None; cur_t = []
        prev = wid
    if cur_e and cur_t: ents[cur_e] = " ".join(cur_t)
    return ents
"""
print("=== app/ml/entity_extractor.py ===")
print(CODE)
print("Usage:")
print("  from app.ml.entity_extractor import extract_entities")
print("  entities = extract_entities('open youtube search trending songs')")
print("  # -> {'query': 'trending songs'}")


In [ ]:
# Cell 13 — Final Benchmark + Summary
import time, statistics

latencies = []
INPUTS = ["open youtube and search trending songs",
          "google VTU results", "send email to boss@company.com"]
for i in range(100):
    s = time.perf_counter()
    extract_entities(INPUTS[i % len(INPUTS)])
    latencies.append((time.perf_counter() - s) * 1000)
latencies.sort()

print(f"Entity Extractor Latency (100 calls):")
print(f"  Median: {statistics.median(latencies):.2f} ms")
print(f"  P95:    {latencies[94]:.2f} ms")
print(f"  Max:    {latencies[-1]:.2f} ms")
print()
print("=" * 55)
print("  MJ Entity Extractor — Training Complete!")
print("  Saved: exports/mj_entity_model/")
print("  Next: open MJ_Intent_Classifier.ipynb")
print("=" * 55)
